In [26]:
import re, traceback

def code_generation_agent_react(prompt: str) -> str:
    """
    Simulated ReACT agent.
    It returns (1) reasoning, (2) buggy code, then (3) fixed code after observing an error.
    """
    print(f"--- Sending Prompt to ReACT Agent ---\n{prompt}\n----------------------------------")
    p = prompt.lower()

    if "reason" in p or "plan" in p:
        return """REASON:
We need a GET request function with robust error handling.
Edge cases: invalid URL, timeout, connection error, HTTP 4xx/5xx, non-JSON response.

PLAN:
1) Write get_data_from_api(api_url)
2) Add timeout and raise_for_status
3) Catch RequestException + JSON errors
4) Return dict on success, None on failure
"""

    if "fix" in p:
        return """```python
import requests

def get_data_from_api(api_url: str):
    \"\"\"Fetch JSON from an API endpoint with robust error handling.\"\"\"
    try:
        response = requests.get(api_url, timeout=5)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.MissingSchema:
        print("Error: Invalid URL format (missing schema like https://).")
        return None
    except requests.exceptions.JSONDecodeError:
        print("Error: Response was not valid JSON.")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return None

# Evidence it runs (no internet needed)
assert get_data_from_api("not a url") is None
print("✅ Fixed code ran and handled invalid URL.")
```"""

    # ACT: intentionally buggy (no error handling) so we can OBSERVE + FIX
    return """```python
import requests

def get_data_from_api(api_url: str):
    response = requests.get(api_url)   # missing timeout + error handling
    return response.json()

# This will error (invalid URL) -> we will OBSERVE and then FIX
print(get_data_from_api("not a url"))
```"""

# 1) REASON
reasoning = code_generation_agent_react("REASON/PLAN: Explain your approach before writing code.")
print(reasoning)

# 2) ACT (CODE)
raw = code_generation_agent_react("ACT: Generate the Python code in one ```python block.")
print("\nMODEL OUTPUT (CODE):\n", raw)

code_block = re.search(r"```python\s*(.*?)```", raw, re.DOTALL).group(1)

# 3) OBSERVE (run code)
print("\n--- EXECUTING GENERATED CODE ---")
try:
    exec(code_block, {})
    print("✅ Generated code ran successfully (unexpected).")
except Exception:
    err = traceback.format_exc()
    print("❌ Execution failed. OBSERVE logs:\n", err)

    # 4) FIX
    fixed_raw = code_generation_agent_react("FIX: Update the code to handle the observed error and add robust handling.")
    print("\nMODEL OUTPUT (FIX):\n", fixed_raw)

    fixed_block = re.search(r"```python\s*(.*?)```", fixed_raw, re.DOTALL).group(1)

    print("\n--- EXECUTING FIXED CODE ---")
    exec(fixed_block, {})

--- Sending Prompt to ReACT Agent ---
REASON/PLAN: Explain your approach before writing code.
----------------------------------
REASON:
We need a GET request function with robust error handling.
Edge cases: invalid URL, timeout, connection error, HTTP 4xx/5xx, non-JSON response.

PLAN:
1) Write get_data_from_api(api_url)
2) Add timeout and raise_for_status
3) Catch RequestException + JSON errors
4) Return dict on success, None on failure

--- Sending Prompt to ReACT Agent ---
ACT: Generate the Python code in one ```python block.
----------------------------------

MODEL OUTPUT (CODE):
 ```python
import requests

def get_data_from_api(api_url: str):
    response = requests.get(api_url)   # missing timeout + error handling
    return response.json()

# This will error (invalid URL) -> we will OBSERVE and then FIX
print(get_data_from_api("not a url"))
```

--- EXECUTING GENERATED CODE ---
❌ Execution failed. OBSERVE logs:
 Traceback (most recent call last):
  File "/tmp/ipython-input-218

Full ReACT Prompt

For this exercise, I structured the prompt using a ReACT approach:

REASON/PLAN: I asked the AI to restate the task and identify possible edge cases, such as invalid URLs, timeouts, connection errors, HTTP 4xx/5xx errors, and non-JSON responses.

ACT (CODE): I instructed the AI to generate a single Python code block defining get_data_from_api(api_url) using the requests library.

OBSERVE: I executed the generated code and reviewed the error logs.

FIX: Based on the observed error, I asked the AI to revise the code and then re-ran the updated version to confirm it worked correctly.

Constraints I included:

The function must include a timeout parameter.

It must use response.raise_for_status().

It must handle RequestException and invalid URL errors.

It must include a small runnable test using asser